# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

# OpenRouter (use for GPT-style models)
openai = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openai_api_key)

# Ollama (local)
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# Model map: label -> (client_key, model_id)
MODEL_MAP = {
    "OpenRouter (GPT)": ("openai", "openai/gpt-4.1-mini"),
    "Ollama (Llama)": ("ollama", "llama3.2:1b"),
}
CLIENTS = {"openai": openai, "ollama": ollama}

OpenAI API Key exists and begins sk-or-v1


In [3]:
system_message = """You are an expert technical tutor. Answer programming and technical questions clearly and concisely. Use examples when helpful. If you don't know, say so."""

In [4]:
# Bonus: glossary tool for looking up technical terms
GLOSSARY = {
    "API": "Application Programming Interface: a set of definitions and protocols for building and integrating software.",
    "REST": "Representational State Transfer: an architectural style for APIs that uses HTTP methods (GET, POST, etc.).",
    "JSON": "JavaScript Object Notation: a lightweight data-interchange format, human-readable and easy to parse.",
    "LLM": "Large Language Model: a type of AI model trained on vast text to generate or understand natural language.",
    "streaming": "Sending data in a continuous flow (e.g. token-by-token) rather than waiting for the full response.",
}

def get_definition(term: str) -> str:
    key = term.strip()
    return GLOSSARY.get(key.upper(), "No definition in glossary.")

get_definition_schema = {
    "name": "get_definition",
    "description": "Look up a technical term in the glossary.",
    "parameters": {
        "type": "object",
        "properties": {
            "term": {"type": "string", "description": "The technical term to look up."},
        },
        "required": ["term"],
        "additionalProperties": False,
    },
}
tools = [{"type": "function", "function": get_definition_schema}]

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_definition":
            args = json.loads(tool_call.function.arguments)
            term = args.get("term", "")
            content = get_definition(term)
            responses.append({
                "role": "tool",
                "content": content,
                "tool_call_id": tool_call.id,
            })
    return responses

In [5]:
def chat(message, history, model_choice):
    if not message or not message.strip():
        yield ""
        return
    client_key, model_id = MODEL_MAP.get(model_choice, ("openai", "openai/gpt-4.1-mini"))
    client = CLIENTS[client_key]
    use_tools = client_key == "openai"

    history_list = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history_list + [{"role": "user", "content": message}]

    if use_tools:
        # OpenRouter with tools: non-streaming tool loop, then yield final content once
        kwargs = {"model": model_id, "messages": messages, "tools": tools, "stream": False}
        while True:
            response = client.chat.completions.create(**kwargs)
            if not response.choices:
                yield ""
                return
            msg = response.choices[0].message
            finish_reason = response.choices[0].finish_reason
            if finish_reason != "tool_calls":
                content = getattr(msg, "content", None) or ""
                if content:
                    yield content
                return
            tool_responses = handle_tool_calls(msg)
            messages.append(msg)
            messages.extend(tool_responses)
            kwargs["messages"] = messages
        return

    # No tools: stream response
    stream = client.chat.completions.create(model=model_id, messages=messages, stream=True)
    result = ""
    for chunk in stream:
        if not chunk.choices:
            continue
        delta = getattr(chunk.choices[0], "delta", None) or {}
        part = getattr(delta, "content", None) or ""
        if part:
            result += part
            yield result

In [6]:
with gr.Blocks(title="Technical Q&A Assistant") as demo:
    gr.Markdown("## Technical Q&A Assistant\nAsk a technical or programming question. Choose the model and get streamed answers.")
    model_dropdown = gr.Dropdown(
        choices=list(MODEL_MAP.keys()),
        value="OpenRouter (GPT)",
        label="Model",
    )
    gr.ChatInterface(
        fn=chat,
        type="messages",
        additional_inputs=[model_dropdown],
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.
